# Implementation of Embedding

### Task
You are given a TEXT and a QUERY.

```
TEXT = ("RAG pipelines retrieve context before generation. "
        "Chunking splits text into overlapping windows. "
        "Embeddings map text into vector space. "
        "Cosine similarity ranks chunks against the query. ")

QUERY = "chunking overlapping windows embeddings"
```

Design a **simple** embedding system that would retrieve relevant chunks from the TEXT related to the QUERY

In [2]:
import numpy as np
import string

# 1. Setup raw variables
TEXT = ("RAG pipelines retrieve context before generation. "
        "Chunking splits text into overlapping windows. "
        "Embeddings map text into vector space. "
        "Cosine similarity ranks chunks against the query. ")

QUERY = "chunking overlapping windows embeddings"

In [3]:
# 1. Split text into words, convert to lowercase, and remove punctuation

words = TEXT.lower().split() # lowercase -> split

translator = str.maketrans('', '', string.punctuation) # remove punctuation

words = [word.translate(translator) for word in words]

words

['rag',
 'pipelines',
 'retrieve',
 'context',
 'before',
 'generation',
 'chunking',
 'splits',
 'text',
 'into',
 'overlapping',
 'windows',
 'embeddings',
 'map',
 'text',
 'into',
 'vector',
 'space',
 'cosine',
 'similarity',
 'ranks',
 'chunks',
 'against',
 'the',
 'query']

In [4]:
# 2. Create chunks with overlapping from words

def chunk_text(word_list, size=8, overlap=3):
    chunks = []
    step = size - overlap
    for i in range(0, len(word_list), step):
        window = word_list[i : i + size]
        chunks.append(" ".join(window))
        if i + size >= len(word_list):
            break
    return chunks

chunks = chunk_text(words, size=8, overlap=3)
chunks

['rag pipelines retrieve context before generation chunking splits',
 'generation chunking splits text into overlapping windows embeddings',
 'overlapping windows embeddings map text into vector space',
 'into vector space cosine similarity ranks chunks against',
 'ranks chunks against the query']

In [5]:
# 3. Create vocabulary for Bag of Words (BoW)
vocabulary = sorted(list(set(words)))
vocabulary

['against',
 'before',
 'chunking',
 'chunks',
 'context',
 'cosine',
 'embeddings',
 'generation',
 'into',
 'map',
 'overlapping',
 'pipelines',
 'query',
 'rag',
 'ranks',
 'retrieve',
 'similarity',
 'space',
 'splits',
 'text',
 'the',
 'vector',
 'windows']

In [11]:
# 4. Generate embeddings (Vectorize texts based on word frequencies)

def vectorize(text_str, vocab):
    text_words = text_str.split()
    return [text_words.count(word) for word in vocab]

chunk_embeddings = [vectorize(c, vocabulary) for c in chunks]
query_embedding = vectorize(QUERY, vocabulary)

print("--- Vocabulary ---")
for v in vocabulary:
    print(v,end=",")

print("\n\n--- Chunks ---")
for idx, c in enumerate(chunks):
    print(f"Chunk {idx}: {c}")

print("\n\n--- Chunk Embeddings ---")
for idx, emb in enumerate(chunk_embeddings):
    print(f"Chunk {idx}: {emb}")


print("\n\n--- Query ---")
print(QUERY)

print("\n--- Query Embedding ---")
print(query_embedding)


--- Vocabulary ---
against,before,chunking,chunks,context,cosine,embeddings,generation,into,map,overlapping,pipelines,query,rag,ranks,retrieve,similarity,space,splits,text,the,vector,windows,

--- Chunks ---
Chunk 0: rag pipelines retrieve context before generation chunking splits
Chunk 1: generation chunking splits text into overlapping windows embeddings
Chunk 2: overlapping windows embeddings map text into vector space
Chunk 3: into vector space cosine similarity ranks chunks against
Chunk 4: ranks chunks against the query


--- Chunk Embeddings ---
Chunk 0: [0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0]
Chunk 1: [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]
Chunk 2: [0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1]
Chunk 3: [1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0]
Chunk 4: [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0]


--- Query ---
chunking overlapping windows e

In [13]:
# 5. Do something obvious: Rank chunks by Cosine Similarity against the QUERY
def cosine_similarity(v1, v2):
    v1_dot_v2 = np.dot(v1, v2)
    norm_1 = np.linalg.norm(v1)
    norm_2 = np.linalg.norm(v2)
    
    return v1_dot_v2 / (norm_1 * norm_2) if norm_1 and norm_2 else 0.0

scores = [cosine_similarity(query_embedding, c_emb) for c_emb in chunk_embeddings]
best_idx = np.argmax(scores)

# Execution Summary
print("--- Extracted Text Chunks ---")
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx}: \"{chunk}\"")

print("\n--- Similarity Match Rankings ---")
for idx, score in enumerate(scores):
    print(f"Chunk {idx} Score: {score:.4f}")

print(f"\n🏆 Top Retrieved Context: \"{chunks[best_idx]}\"")


--- Extracted Text Chunks ---
Chunk 0: "rag pipelines retrieve context before generation chunking splits"
Chunk 1: "generation chunking splits text into overlapping windows embeddings"
Chunk 2: "overlapping windows embeddings map text into vector space"
Chunk 3: "into vector space cosine similarity ranks chunks against"
Chunk 4: "ranks chunks against the query"

--- Similarity Match Rankings ---
Chunk 0 Score: 0.1768
Chunk 1 Score: 0.7071
Chunk 2 Score: 0.5303
Chunk 3 Score: 0.0000
Chunk 4 Score: 0.0000

🏆 Top Retrieved Context: "generation chunking splits text into overlapping windows embeddings"
